In [2]:
import warnings
from mcp.mcp_runner import MCPRunner
# Importa a classe específica da estratégia que criamos
from strats.flight_to_quality import FlightToQualityStrategy 

def run_first_strategy_test():
    """
    Carrega e testa a estratégia 'FlightToQuality' (VIX -> BTC/XAU).
    """
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    warnings.filterwarnings('ignore', category=UserWarning)

    print("--- Iniciando Teste da Estratégia 'Flight to Quality' ---")
    
    # 1. Definir os caminhos dos dados
    # (Incluindo os novos arquivos, mesmo que esta estratégia não os use,
    # o runner.load_data() vai uni-los)
    data_paths = {
        'btc': 'data/BTC_com_ciclos_HALVING.csv',
        'xau': 'data/XAU-USD.csv',
        'ng': 'data/Gas Natural Futures.csv',
        'macro': 'data/macro_data.csv',
        'spy': 'data/SPY.csv',
        'idy': 'data/IDY.csv'
    }

    # 2. Instanciar a Estratégia
    strategy_to_test = FlightToQualityStrategy()

    # 3. Instanciar o Runner
    runner = MCPRunner(data_paths=data_paths, strategy=strategy_to_test)

    # 4. Carregar e unificar os dados
    try:
        # Filtra para um período onde todos os dados existem
        runner.load_data(start_year=2018, end_year=2024)
        print("\nDados carregados e unificados com sucesso.")
        print(f"Colunas disponíveis para a estratégia: {runner.data.columns.to_list()}")
    except Exception as e:
        print(f"\n❌ ERRO ao carregar dados: {e}")
        print("Verifique os nomes das colunas e caminhos dos arquivos.")
        return
# ... (imports e setup são os mesmos) ...

    # 5. Rodar o Teste de Permutação (MACRO)
    print("\nIniciando Teste de Permutação Macro In-Sample...")
    
    try:
        # --- NOVO: Você pode escolher a métrica para o P-Value ---
        # O padrão é 'profit_factor'. Vamos testar o 'sharpe'
        
        results = runner.run_macro_insample_MCP(
            n_permutations=2000,
            stat_to_test='sharpe'  # <-- NOVO! Pode ser 'sharpe', 'sortino', 'calmar', etc.
        )
        
        # O print dos resultados agora é automático (dentro do runner)
        # Não precisa mais do bloco "--- Resultados do Teste ---" aqui.

        # 6. Plotar os resultados
        print("\nGerando gráficos...")
        
        # Plot 1: Histograma do Sharpe (o que testamos)
        runner.plot_results(
            results, 
            test_type="MCP Macro (Sharpe)", 
            metric_to_plot="sharpe"
        )
        
        # Plot 2: Histograma do Profit Factor (só para ver)
        runner.plot_results(
            results, 
            test_type="MCP Macro (Profit Factor)", 
            metric_to_plot="profit_factor"
        )

    except Exception as e:
        print(f"\n❌ ERRO durante a execução do teste: {e}")
        import traceback
        traceback.print_exc()

# ... (resto do script) ...

# --- Ponto de entrada para executar o script ---
if __name__ == "__main__":
    run_first_strategy_test()

--- Iniciando Teste da Estratégia 'Flight to Quality' ---
Carregando e unificando dados...
  ✅ data/BTC_com_ciclos_HALVING.csv carregado.
  ✅ data/XAU-USD.csv carregado.
  ✅ data/Gas Natural Futures.csv carregado.
  ✅ data/macro_data.csv carregado.
    ⚠️ Aviso: data/SPY.csv tem 251 datas duplicadas. Removendo (mantendo a última)...
  ✅ data/SPY.csv carregado.
  ✅ data/IDY.csv carregado.

Unificando DataFrames (join='inner')...
Preenchendo dados (ffill)...
Limpando NaNs restantes no início do histórico...

Dados unificados carregados. 0 linhas.
❌ ERRO: DataFrame final está vazio. Verifique os dados de entrada e o intervalo de datas.
Colunas disponíveis: ['btc_price', 'ciclo_halving', 'xau', 'open_xau', 'high_xau', 'low_xau', 'Vol.', 'Var%', 'ng', 'open_ng', 'high_ng', 'low_ng', 'Vol.', 'Var%', 'vix', 'juros_10a', 'dolar', 'petroleo', 'spy', 'open_spy', 'high_spy', 'low_spy', 'Vol.', 'Var%', 'idy', 'open_idy', 'high_idy', 'low_idy', 'Vol.', 'Var%']

Dados carregados e unificados com suc

Traceback (most recent call last):
  File "C:\Users\kraus\AppData\Local\Temp\ipykernel_37396\338484575.py", line 52, in run_first_strategy_test
    results = runner.run_macro_insample_MCP(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kraus\projetos - git\quant\mcp\mcp_runner.py", line 187, in run_macro_insample_MCP
    raise ValueError("Dados não carregados ou DataFrame vazio. Chame load_data() primeiro.")
ValueError: Dados não carregados ou DataFrame vazio. Chame load_data() primeiro.


In [1]:
import warnings
import pandas as pd # Adicionado para o diagnóstico
from mcp.mcp_runner import MCPRunner
from strats.macro_regime import MacroRegimeStrategy 

# ===================================================================
# --- INÍCIO DO PROCESSO DE DIAGNÓSTICO (O QUE VOCÊ PEDIU) ---
# ===================================================================

def run_diagnostic_check(data_paths_to_check):
    """
    Roda uma verificação de datas nos arquivos CSV para garantir
    que o parse está funcionando antes de rodar o runner.
    """
    print("\n" + "#"*80)
    print("--- INICIANDO PROCESSO DE DIAGNÓSTICO DE DATAS ---")
    print("#"*80 + "\n")
    
    # Esta lógica DEVE ser idêntica à de mcp_runner.py
    semicolon_files = ['XAU-USD', 'Gas Natural', 'SPY.csv', 'IDY.csv', 'BTC.csv']
    
    for name, file_path in data_paths_to_check.items():
        print(f"--- Inspecionando: {name} ({file_path}) ---")
        try:
            is_semicolon = any(n in file_path for n in semicolon_files)
            
            # 1. Lê as 5 primeiras linhas como texto
            df_raw = pd.read_csv(file_path, 
                                 delimiter=';' if is_semicolon else ',', 
                                 dtype=str, 
                                 nrows=5)
            
            print(f"Amostra de dados (como texto):")
            print(df_raw.to_string())

            # 2. Identifica a coluna de índice
            index_col_name = None
            if 'Data' in df_raw.columns: index_col_name = 'Data'
            elif 'Date' in df_raw.columns: index_col_name = 'Date'
            else: index_col_name = df_raw.columns[0]
            
            print(f"\nUsando coluna de índice: '{index_col_name}'")
            print(f"Amostra de datas (string): {df_raw[index_col_name].values}")

            # 3. Tenta a conversão de data (a lógica EXATA do mcp_runner.py)
            date_format = '%d.%m.%Y' if is_semicolon else None
            print(f"Lógica de parse: is_semicolon={is_semicolon}, format='{date_format}'")

            # 4. Recarrega o arquivo todo para aplicar e ver o resultado
            df_full = pd.read_csv(file_path, 
                                  delimiter=';' if is_semicolon else ',', 
                                  dtype=str)
            df_full = df_full.set_index(index_col_name)
            df_full.index = pd.to_datetime(df_full.index, 
                                           errors='coerce', 
                                           format=date_format) # A correção chave
            
            df_full = df_full[df_full.index.notna()] # Limpa datas inválidas

            if len(df_full) == 0:
                print("❌ DIAGNÓSTICO: ERRO. O parse de data falhou. 0 linhas válidas.")
            else:
                print("✅ DIAGNÓSTICO: SUCESSO. Datas lidas corretamente.")
                print(f"  -> Data de Início (Parsed): {df_full.index.min()}")
                print(f"  -> Data de Fim (Parsed):    {df_full.index.max()}")
        
        except Exception as e:
            print(f"❌ ERRO NO DIAGNÓSTICO: Falha ao ler {file_path}: {e}")
        print("-"*80 + "\n")

# ===================================================================
# --- FIM DO PROCESSO DE DIAGNÓSTICO ---
# ===================================================================


def run_strategy_horse_race():
    """
    Roda a "corrida de cavalos".
    """
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    warnings.filterwarnings('ignore', category=UserWarning)
    warnings.filterwarnings('ignore', category=FutureWarning)

    # --- CAMINHOS DE DADOS GLOBAIS (APENAS PARA REFERÊNCIA) ---
    all_data_paths = {
        'btc': 'data/BTC_com_ciclos_HALVING.csv',
        'xau': 'data/XAU-USD.csv',
        'ng': 'data/Gas Natural Futures.csv',
        'macro': 'data/macro_data.csv',
        'spy': 'data/SPY.csv',
        'idy': 'data/IDY.csv'
    }

    # --- CHAMA O DIAGNÓSTICO PRIMEIRO ---
    # Vamos checar um arquivo de cada tipo (um com ; e um com ,)
    diagnostic_paths = {
        'SPY (Exemplo ;)': all_data_paths['spy'],
        'Macro (Exemplo ,)': all_data_paths['macro'],
        'XAU (Exemplo ;)': all_data_paths['xau'],
        'BTC (Exemplo ,)': all_data_paths['btc']
    }
    run_diagnostic_check(diagnostic_paths)
    
    print("\n" + "="*80)
    print("--- INICIANDO TESTES DA ESTRATÉGIA ---")
    print("="*80 + "\n")
    # --- FIM DA CHAMADA DE DIAGNÓSTICO ---


    # ===================================================================
    # --- TESTE A: Clássica (SPY -> IDY) ---
    # ===================================================================
    print("\n" + "="*80)
    print("--- EXECUTANDO TESTE: A: Clássica (SPY -> IDY) ---")
    print("="*80)
    
    strategy_A = MacroRegimeStrategy()
    strategy_A.configure(
        signal_column='vix',
        weights_risk_on={'spy_ret': 1.0},
        weights_risk_off={'idy_ret': 1.0}
    )
    
    paths_A = {
        'spy': all_data_paths['spy'],
        'idy': all_data_paths['idy'],
        'macro': all_data_paths['macro'] 
    }

    try:
        runner_A = MCPRunner(data_paths=paths_A, strategy=strategy_A)
        runner_A.load_data(start_year=2018, end_year=2024) # Isso vai falhar se o diag falhar
        
        results_A = runner_A.run_macro_insample_MCP(
            n_permutations=2000,
            stat_to_test='sharpe'
        )
        
        print(f"\n--- GRÁFICOS PARA: A: Clássica (SPY -> IDY) ---")
        runner_A.plot_results(results_A, test_type="MCP (A: Clássica)", metric_to_plot="sharpe")

    except Exception as e:
        print(f"\n❌ ERRO durante a execução do teste 'A: Clássica': {e}")


    # ===================================================================
    # --- TESTE B: Benchmark (60/40 Passivo) ---
    # ===================================================================
    print("\n" + "="*80)
    print("--- EXECUTANDO TESTE: B: Benchmark (60/40 Passivo) ---")
    print("="*80)
    
    strategy_B = MacroRegimeStrategy()
    strategy_B.configure(
        signal_column='vix',
        weights_risk_on={'spy_ret': 0.6, 'idy_ret': 0.4},
        weights_risk_off={'spy_ret': 0.6, 'idy_ret': 0.4}
    )
    
    paths_B = {
        'spy': all_data_paths['spy'],
        'idy': all_data_paths['idy'],
        'macro': all_data_paths['macro']
    }

    try:
        runner_B = MCPRunner(data_paths=paths_B, strategy=strategy_B)
        runner_B.load_data(start_year=2018, end_year=2024)
        
        results_B = runner_B.run_macro_insample_MCP(
            n_permutations=2000,
            stat_to_test='sharpe'
        )
        
        print(f"\n--- GRÁFICOS PARA: B: Benchmark (60/40 Passivo) ---")
        runner_B.plot_results(results_B, test_type="MCP (B: Benchmark)", metric_to_plot="sharpe")

    except Exception as e:
        print(f"\n❌ ERRO durante a execução do teste 'B: Benchmark': {e}")


    # ===================================================================
    # --- TESTE C: Original (BTC -> XAU) ---
    # ===================================================================
    print("\n" + "="*80)
    print("--- EXECUTANDO TESTE: C: Original (BTC -> XAU) ---")
    print("="*80)

    strategy_C = MacroRegimeStrategy()
    strategy_C.configure(
        signal_column='vix',
        weights_risk_on={'btc_ret': 0.5, 'xau_ret': 0.5},
        weights_risk_off={'xau_ret': 1.0}
    )

    paths_C = {
        'btc': all_data_paths['btc'],
        'xau': all_data_paths['xau'],
        'macro': all_data_paths['macro']
    }

    try:
        runner_C = MCPRunner(data_paths=paths_C, strategy=strategy_C)
        runner_C.load_data(start_year=2018, end_year=2024)
        
        results_C = runner_C.run_macro_insample_MCP(
            n_permutations=2000,
            stat_to_test='sharpe'
        )
        
        print(f"\n--- GRÁFICOS PARA: C: Original (BTC -> XAU) ---")
        runner_C.plot_results(results_C, test_type="MCP (C: Original)", metric_to_plot="sharpe")

    except Exception as e:
        print(f"\n❌ ERRO durante a execução do teste 'C: Original': {e}")


if __name__ == "__main__":
    run_strategy_horse_race()


################################################################################
--- INICIANDO PROCESSO DE DIAGNÓSTICO DE DATAS ---
################################################################################

--- Inspecionando: SPY (Exemplo ;) (data/SPY.csv) ---
Amostra de dados (como texto):
         Data  Último Abertura  Máxima  Mínima    Vol.    Var%
0  31.10.2025  682,06   685,04  685,08  679,24  87,16M   0,33%
1  30.10.2025  679,83    683,9  685,94  679,83  76,34M  -1,10%
2  29.10.2025  687,39   688,72   689,7  682,87  85,36M   0,05%
3  28.10.2025  687,06   687,05  688,91  684,83  61,74M   0,27%
4  27.10.2025  685,24   682,73  685,54  682,11  63,34M   1,18%

Usando coluna de índice: 'Data'
Amostra de datas (string): ['31.10.2025' '30.10.2025' '29.10.2025' '28.10.2025' '27.10.2025']
Lógica de parse: is_semicolon=True, format='%d.%m.%Y'
✅ DIAGNÓSTICO: SUCESSO. Datas lidas corretamente.
  -> Data de Início (Parsed): 1993-02-01 00:00:00
  -> Data de Fim (Parsed):    2025-10-31 

In [1]:
import pandas as pd
import numpy as np
import os
from typing import Dict, Any, List, Set, Optional
from collections import namedtuple
from tqdm import tqdm
from itertools import product

# --- Definição das Classes Base (para funcionar em um só arquivo) ---

StrategyResult = namedtuple('StrategyResult', ['signal', 'metadata'])
OptimisationResult = namedtuple('OptimisationResult', ['best_params', 'best_score', 'all_results'])

class TradingStrategy:
    """ Classe base simplificada """
    def __init__(self, name="BaseStrategy"):
        self.name = name

    def generate_signal(self, ohlc: pd.DataFrame, **params) -> StrategyResult:
        raise NotImplementedError

    def get_parameter_space(self) -> Dict[str, List[Any]]:
        return {}

    def _generate_param_combinations(self, param_space: Dict[str, List[Any]]) -> List[Dict[str, Any]]:
        keys = param_space.keys()
        values = param_space.values()
        combinations = list(product(*values))
        return [dict(zip(keys, combo)) for combo in combinations]

    def _calculate_score(self, result: StrategyResult, price_returns: pd.Series, score_func: str) -> float:
        # Versão base (que está errada para portfólios)
        print("!! Usando _calculate_score da CLASSE BASE (ERRADO) !!")
        return 0.0

    def optimise(self, ohlc: pd.DataFrame, score_func: str = 'profit_factor') -> OptimisationResult:
        # Versão base (que está errada para portfólios)
        print("!! Usando optimise da CLASSE BASE (ERRADO) !!")
        return OptimisationResult({}, 0.0, [])


# --- Classe da Estratégia (COM A CORREÇÃO) ---

class MacroRegimeStrategy(TradingStrategy):
    """
    Sua classe 'MacroRegimeStrategy' com as correções de _calculate_score e optimise.
    """
    
    def __init__(self):
        super().__init__(name="FlexibleMacroRegimeStrategy")
        self.weights_risk_on = {}
        self.weights_risk_off = {}
        self.signal_column = 'vix'
    
    def configure(self, signal_column: str = 'vix',
                  weights_risk_on: Dict[str, float] = None,
                  weights_risk_off: Dict[str, float] = None):
        self.signal_column = signal_column
        if weights_risk_on is not None: self.weights_risk_on = weights_risk_on
        if weights_risk_off is not None: self.weights_risk_off = weights_risk_off
        if not self.weights_risk_on or not self.weights_risk_off:
            raise ValueError("Pesos não configurados.")
    
    def _get_required_assets(self) -> Set[str]:
        assets = set()
        assets.update(self.weights_risk_on.keys())
        assets.update(self.weights_risk_off.keys())
        return assets
    
    def _calculate_asset_returns(self, df: pd.DataFrame, assets: Set[str]) -> pd.DataFrame:
        for ret_col in assets:
            asset_col = ret_col.replace('_ret', '')
            if asset_col not in df.columns:
                raise ValueError(f"Coluna de preço '{asset_col}' não encontrada.")
            
            df[asset_col] = pd.to_numeric(df[asset_col], errors='coerce')
            valid_prices = df[asset_col].notna() & (df[asset_col] > 0)
            if valid_prices.sum() < 2:
                raise ValueError(f"Coluna '{asset_col}' não tem dados válidos.")
            
            df[ret_col] = np.log(df[asset_col]).diff()
        return df
    
    def generate_signal(self, ohlc: pd.DataFrame, **params) -> StrategyResult:
        df = ohlc.copy()
        signal_quantile = params.get('signal_quantile', 0.7)
        
        if not self.weights_risk_on or not self.weights_risk_off:
            raise ValueError("Estratégia não configurada!")
        
        required_assets = self._get_required_assets()
        df = self._calculate_asset_returns(df, required_assets)
        
        if self.signal_column not in df.columns:
            raise ValueError(f"Sinal '{self.signal_column}' não encontrado.")
        if df[self.signal_column].isna().all():
            raise ValueError(f"Coluna de sinal '{self.signal_column}' só tem NaNs.")
        
        threshold = df[self.signal_column].quantile(signal_quantile)
        df['regime'] = np.where(df[self.signal_column] > threshold, 'Risk-Off', 'Risk-On')
        
        def strategy_return(row):
            w = self.weights_risk_off if row['regime'] == 'Risk-Off' else self.weights_risk_on
            ret = 0.0
            for asset_ret_col, weight in w.items():
                if weight == 0: continue
                asset_ret_value = row[asset_ret_col]
                if pd.notna(asset_ret_value):
                    ret += asset_ret_value * weight
            return ret
        
        strat_log_ret = df.apply(strategy_return, axis=1)
        
        metadata = {
            'signal_is_returns': True,
            'signal_column': self.signal_column,
            'signal_quantile': signal_quantile,
            'threshold': threshold,
        }
        return StrategyResult(signal=strat_log_ret, metadata=metadata)
    
    def get_parameter_space(self) -> Dict[str, List[Any]]:
        return {'signal_quantile': [0.65, 0.7, 0.75, 0.8, 0.85]}

    # --- MÉTODOS DE CORREÇÃO ---

    def _calculate_score(self, result: StrategyResult, price_returns: Optional[pd.Series], score_func: str) -> float:
        """
        Sobrescrita: 'result.signal' JÁ SÃO os retornos do portfólio.
        O 'price_returns' é ignorado.
        """
        strategy_returns = result.signal.fillna(0)
        
        if score_func == 'profit_factor':
            positive_returns = strategy_returns[strategy_returns > 0]
            negative_returns = strategy_returns[strategy_returns < 0]
            if len(negative_returns) == 0:
                pf = np.inf if len(positive_returns) > 0 else 0.0
            else:
                pf = positive_returns.sum() / negative_returns.abs().sum()
            return pf if (pd.notna(pf) and np.isfinite(pf)) else 0.0

        elif score_func == 'sharpe':
            if strategy_returns.std() > 1e-9:
                sharpe = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252)
                return sharpe if pd.notna(sharpe) else 0.0
            return 0.0
        else:
            raise ValueError(f"Função de pontuação desconhecida: {score_func}")

    def optimise(self, ohlc: pd.DataFrame, score_func: str = 'profit_factor') -> OptimisationResult:
        """
        Sobrescrita: Otimização para um portfólio.
        """
        param_space = self.get_parameter_space()
        best_score = float('-inf')
        best_params = {}
        all_results = []

        param_combinations = self._generate_param_combinations(param_space)
        print(f"Otimizando {len(param_combinations)} combinações para {self.name}...")

        for params in tqdm(param_combinations, desc="Otimizando parâmetros"):
            try:
                result = self.generate_signal(ohlc, **params) 
                score = self._calculate_score(result, None, score_func) 
                all_results.append((params.copy(), score))
                if score > best_score:
                    best_score = score
                    best_params = params.copy()
            except Exception as e:
                continue
        return OptimisationResult(best_params, best_score, all_results)

# --- FUNÇÃO PARA CRIAR DADOS FALSOS ---

def create_fake_data():
    """ Cria arquivos CSV falsos, mas ALINHADOS, para teste. """
    print("Criando arquivos CSV falsos para o teste...")
    dates = pd.date_range(start='2018-01-01', end='2024-12-31', freq='B')
    n = len(dates)
    
    # SPY (spy)
    spy_price = (np.random.randn(n).cumsum() * 0.1) + 100
    df_spy = pd.DataFrame({'Date': dates, 'spy': spy_price})
    df_spy.to_csv('SPY.csv', index=False)
    
    # IDY (idy)
    idy_price = (np.random.randn(n).cumsum() * 0.05) + 50
    df_idy = pd.DataFrame({'Date': dates, 'idy': idy_price})
    df_idy.to_csv('IDY.csv', index=False)
    
    # Macro (vix)
    vix_val = np.random.uniform(10, 40, size=n)
    df_macro = pd.DataFrame({'Date': dates, 'vix': vix_val})
    df_macro.to_csv('macro_data.csv', index=False)
    
    print("Arquivos falsos criados: SPY.csv, IDY.csv, macro_data.csv")

# --- RUNNER SIMPLIFICADO ---

class SimpleMCPRunner:
    """
    Um MCPRunner simplificado que foca em carregar dados e
    calcular as métricas FINAIS Corretamente.
    """
    def __init__(self, data_paths, strategy):
        self.data_paths = data_paths
        self.strategy = strategy
        self.ohlc = None
    
    def load_data(self, start_year, end_year):
        print("Carregando dados...")
        try:
            df_spy = pd.read_csv(self.data_paths['spy'], parse_dates=['Date'], index_col='Date')
            df_idy = pd.read_csv(self.data_paths['idy'], parse_dates=['Date'], index_col='Date')
            df_macro = pd.read_csv(self.data_paths['macro'], parse_dates=['Date'], index_col='Date')
        except FileNotFoundError as e:
            print(f"Erro: {e}. Arquivos de dados falsos não encontrados.")
            print("Execute create_fake_data() primeiro.")
            return

        # *** ESTA É UMA CAUSA PROVÁVEL DE ERRO NO SEU CÓDIGO REAL ***
        # Usamos 'inner' join para garantir alinhamento. Se seus dados
        # reais não se sobrepõem, o DataFrame fica vazio.
        self.ohlc = pd.concat([df_spy, df_idy, df_macro], axis=1, join='inner')
        
        # Filtra pelo período
        self.ohlc = self.ohlc.loc[f'{start_year}-01-01':f'{end_year}-12-31']
        
        # Remove quaisquer NaNs (ex: feriados)
        self.ohlc = self.ohlc.ffill().dropna()
        
        if len(self.ohlc) == 0:
            raise Exception("load_data resultou em DataFrame vazio. "
                            "Verifique o alinhamento das datas nos seus CSVs!")
        
        print(f"Dados carregados e unificados com sucesso. {len(self.ohlc)} linhas.")

    def calculate_final_metrics(self, strategy_returns):
        """
        Esta é a função que seu MCPRunner real parece estar
        executando incorretamente (com lógica antiga).
        Esta versão é a CORRETA.
        """
        rets = strategy_returns.fillna(0)
        
        if rets.std() < 1e-9 or rets.abs().sum() < 1e-9:
            print("AVISO: Retornos da estratégia são zero ou constantes.")
            return {
                'total_ret': 0.0, 'ann_ret': 0.0, 'vol': 0.0,
                'sharpe': 0.0, 'profit_factor': 0.0
            }

        total_ret = (np.exp(rets.sum()) - 1) * 100
        ann_ret = (np.exp(rets.mean() * 252) - 1) * 100
        vol = (rets.std() * np.sqrt(252)) * 100
        
        sharpe = (rets.mean() / rets.std()) * np.sqrt(252)
        
        pos_ret = rets[rets > 0].sum()
        neg_ret = rets[rets < 0].abs().sum()
        
        profit_factor = pos_ret / neg_ret if neg_ret > 0 else np.inf
        
        return {
            'total_ret': total_ret,
            'ann_ret': ann_ret,
            'vol': vol,
            'sharpe': sharpe,
            'profit_factor': profit_factor
        }

    def run_macro_insample_MCP(self, n_permutations, stat_to_test):
        """
        O loop de teste principal.
        """
        print("\nOtimizando parâmetros...")
        opt_result = self.strategy.optimise(self.ohlc, score_func='profit_factor')
        
        print(f"Parâmetros otimizados: {opt_result.best_params}")
        print(f"Score In-sample (profit_factor): {opt_result.best_score:.4f}")
        
        if opt_result.best_score == 0.0:
            print("❌ ERRO: O score da otimização foi 0.0. "
                  "Isso sugere que 'generate_signal' está retornando zeros.")
            # Não pare, vamos ver as métricas finais.

        # Gera o sinal real com os melhores parâmetros
        real_result = self.strategy.generate_signal(self.ohlc, **opt_result.best_params)

        # --- ESTA É A PARTE CRÍTICA QUE ESTÁ FALHANDO NO SEU CÓDIGO ---
        # Seu runner real está chamando uma função de métrica antiga.
        # Ele DEVE chamar uma função que entenda 'real_result.signal'
        
        print("\n" + "="*50)
        print("--- Métricas da Estratégia Real (Otimizada) ---")
        print("="*50)
        
        # Usamos nossa função de métrica *correta*
        metrics = self.calculate_final_metrics(real_result.signal)
        
        print(f"  Retorno Total (%):        {metrics['total_ret']:.2f}")
        print(f"  Retorno Anualizado (%):   {metrics['ann_ret']:.2f}")
        print(f"  Volatilidade (%):         {metrics['vol']:.2f}")
        print(f"  Profit Factor:            {metrics['profit_factor']:.4f}")
        print(f"  Sharpe Ratio:             {metrics['sharpe']:.4f}")
        
        print("\n--- Resultado do Teste de Permutação (Simulado) ---")
        print(f"  Métrica Testada:          {stat_to_test}")
        
        if metrics[stat_to_test] <= 0:
             print("  P-Value:                  1.0000 (Métrica real é zero ou neg)")
             print("  ❌ Resultado: Não Significante.")
        else:
             print("  P-Value:                  0.0123 (Valor Fixo para Exemplo)")
             print("  ✅ Resultado: Significante (Exemplo).")

        print("="*50)


# --- FUNÇÃO PRINCIPAL DE TESTE ---

def run_test():
    # 1. Criar os dados falsos
    create_fake_data()

    # 2. Definir caminhos (mesmo estando na mesma pasta)
    data_paths = {
        'spy': 'SPY.csv',
        'idy': 'IDY.csv',
        'macro': 'macro_data.csv'
    }

    # 3. Configurar a Estratégia A
    strategy_A = MacroRegimeStrategy()
    strategy_A.configure(
        signal_column='vix',
        weights_risk_on={'spy_ret': 1.0},
        weights_risk_off={'idy_ret': 1.0}
    )

    # 4. Configurar e rodar o Runner
    print("\n" + "="*80)
    print("--- EXECUTANDO TESTE MRE: A: Clássica (SPY -> IDY) ---")
    print("="*80)
    
    runner = SimpleMCPRunner(data_paths=data_paths, strategy=strategy_A)
    
    try:
        runner.load_data(start_year=2018, end_year=2024)
        
        runner.run_macro_insample_MCP(
            n_permutations=2000, # (Não usado neste MRE, apenas simulado)
            stat_to_test='sharpe'
        )
        
    except Exception as e:
        print(f"\n❌ ERRO DURANTE A EXECUÇÃO DO TESTE MRE: {e}")
        import traceback
        traceback.print_exc()

    # Limpar arquivos falsos
    os.remove('SPY.csv')
    os.remove('IDY.csv')
    os.remove('macro_data.csv')
    print("\nArquivos de teste limpos.")

if __name__ == "__main__":
    run_test()

Criando arquivos CSV falsos para o teste...
Arquivos falsos criados: SPY.csv, IDY.csv, macro_data.csv

--- EXECUTANDO TESTE MRE: A: Clássica (SPY -> IDY) ---
Carregando dados...
Dados carregados e unificados com sucesso. 1827 linhas.

Otimizando parâmetros...
Otimizando 5 combinações para FlexibleMacroRegimeStrategy...


Otimizando parâmetros: 100%|██████████| 5/5 [00:00<00:00, 29.79it/s]

Parâmetros otimizados: {'signal_quantile': 0.7}
Score In-sample (profit_factor): 1.1158

--- Métricas da Estratégia Real (Otimizada) ---
  Retorno Total (%):        7.87
  Retorno Anualizado (%):   1.05
  Volatilidade (%):         1.52
  Profit Factor:            1.1158
  Sharpe Ratio:             0.6890

--- Resultado do Teste de Permutação (Simulado) ---
  Métrica Testada:          sharpe
  P-Value:                  0.0123 (Valor Fixo para Exemplo)
  ✅ Resultado: Significante (Exemplo).

Arquivos de teste limpos.
